In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import optuna
from optuna.samplers import TPESampler
import joblib

# Load the data
data = pd.read_csv('preprocessed.csv')

# Define features and target
features = ['current_stop_name', 'next_stop_name', 'day_of_week', 'is_holiday', 
            'is_peak_hour', 'weather_condition', 'passenger_count', 'current_speed', 
            'distance_to_next_stop', 'current_lat', 'current_lon']
target = 'eta_minutes'

X = data[features]
y = data[target]

# Convert boolean columns to int
X['is_holiday'] = X['is_holiday'].astype(int)
X['is_peak_hour'] = X['is_peak_hour'].astype(int)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Function to evaluate model
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2 Score: {r2:.4f}")
    
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

# Baseline Random Forest model
print("Training baseline Random Forest model...")
baseline_model = RandomForestRegressor(random_state=42)
baseline_model.fit(X_train, y_train)

print("\nBaseline Model Performance:")
baseline_metrics = evaluate_model(baseline_model, X_test, y_test)

# Optuna optimization
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
    }
    
    model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    
    return rmse

print("\nStarting Optuna optimization...")
study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=50)

# Train optimized model
print("\nTraining optimized model with best parameters...")
best_params = study.best_params
optimized_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
optimized_model.fit(X_train, y_train)

print("\nOptimized Model Performance:")
optimized_metrics = evaluate_model(optimized_model, X_test, y_test)

# Compare baseline and optimized models
print("\nPerformance Comparison:")
print(f"Baseline RMSE: {baseline_metrics['RMSE']:.4f}")
print(f"Optimized RMSE: {optimized_metrics['RMSE']:.4f}")
print(f"Improvement: {(baseline_metrics['RMSE'] - optimized_metrics['RMSE']):.4f} ({((baseline_metrics['RMSE'] - optimized_metrics['RMSE'])/baseline_metrics['RMSE']*100):.2f}%)")

# Save the optimized model
model_filename = 'optimized_randomforest_eta_predictor.pkl'
joblib.dump(optimized_model, model_filename)
print(f"\nOptimized model saved as {model_filename}")

# Feature importance
importance = pd.DataFrame({
    'feature': features,
    'importance': optimized_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(importance)

c:\Users\cheng\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\cheng\AppData\Local\Temp\ipykernel_35200\3487010628.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['is_holiday'] = X['is_holiday'].astype(int)
C:\Users\cheng\AppData\Local\Temp\ipykernel_35200\3487010628.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexin

Training baseline Random Forest model...

Baseline Model Performance:


[I 2025-04-11 17:20:45,888] A new study created in memory with name: no-name-78e0151e-4913-49a2-901a-9a6a2239cd69


MAE: 0.3230
MSE: 0.3108
RMSE: 0.5575
R2 Score: 0.9627

Starting Optuna optimization...


c:\Users\cheng\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
[I 2025-04-11 17:20:48,362] Trial 0 finished with value: 0.5571179307770671 and parameters: {'n_estimators': 437, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: 0.5571179307770671.
c:\Users\cheng\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
[I 2025-04-11 17:20:50,182] Trial 1 finished with value: 0.9357313730660595 and parameters: {'n_estimators': 737, 'max_depth': 5, 'min_samples_


Training optimized model with best parameters...

Optimized Model Performance:
MAE: 0.3142
MSE: 0.2859
RMSE: 0.5347
R2 Score: 0.9657

Performance Comparison:
Baseline RMSE: 0.5575
Optimized RMSE: 0.5347
Improvement: 0.0228 (4.10%)

Optimized model saved as optimized_randomforest_eta_predictor.pkl

Feature Importance:
                  feature  importance
7           current_speed    0.766478
8   distance_to_next_stop    0.128731
0       current_stop_name    0.080417
4            is_peak_hour    0.012550
9             current_lat    0.006034
1          next_stop_name    0.001811
10            current_lon    0.001720
6         passenger_count    0.001322
2             day_of_week    0.000480
5       weather_condition    0.000323
3              is_holiday    0.000133
